# timeseries, correcting for rectification

## Imports

In [ ]:
import src.utils
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import sklearn.linear_model
import pathlib
import os
import sklearn.linear_model

## RNG
rng = np.random.default_rng()

## color palette
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## paths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
# SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## compute indices

In [ ]:
def get_nino3(x, z_t=70, lat_bound=5):
    """get Niño 3 average, with checks in place"""

    if "z_t" in x.dims:
        x = x.sel(z_t=z_t, method="nearest")

    if "latitude" in x.dims:
        x = x.sel(latitude=slice(-lat_bound, lat_bound)).mean("latitude")

    return x.sel(longitude=slice(210, 270)).mean("longitude")


def get_nino3_helper(x, lat_bounds):
    """get Niño 3 average, with checks in place"""

    ## get index to select data
    idx = dict(latitude=slice(*lat_bounds), longitude=slice(210, 270))

    return x.sel(idx).mean(["longitude", "latitude"])


def get_w_sub(w, mld=70, delta=20):
    """get velocity at base of mixed layer"""

    ## get temperature difference
    w_sub = w.sel(z_t=slice(mld, mld + delta)).mean("z_t")

    return w_sub


def get_dTdz_sub(Tsub, mld=70, delta=20):
    """get difference between surface and subsurface temp"""

    ## get temperature difference
    dT = src.utils.get_dT_sub(Tsub=Tsub, Hm=mld, delta=delta)

    ## get gradient
    dTdz = -dT / mld

    return dTdz


def get_dTdz_sub2(Tsub, mld=80):
    """get difference between surface and subsurface temp"""

    ## get temperature difference
    dT = Tsub.sel(z_t=mld, method="nearest") - Tsub.sel(z_t=0, method="nearest")

    ## get gradient
    dTdz = -dT / mld

    return dTdz

### individual metrics

#### Niño 3 averages

In [ ]:
## should we use "wide" data?
USE_WIDE = False

## load spatial data
forced, anom_ = src.utils.load_consolidated()

if USE_WIDE:

    ## load wide data
    forced_05, anom_05 = src.utils.load_consolidated_05()

    for v in list(forced_05):
        forced[v] = forced_05[v]
        anom_[v] = anom_05[v]

## get subset of data for total
VARNAMES = ["T", "w", "sst", "taux"]
total = anom_[VARNAMES] + forced[VARNAMES]
total = xr.merge([forced[[f"{v}_comp" for v in VARNAMES]], total])

In [ ]:
sel_n3 = lambda x: x.sel(longitude=slice(210, 270)).mean("longitude")

## compute stratification
dTdz_n3 = src.utils.reconstruct_wrapper(
    total[["T", "T_comp"]],
    lambda x: get_dTdz_sub(sel_n3(x)),
    # lambda x: get_dTdz_sub2(x.sel(longitude=slice(210, 270)).mean("longitude")),
)["T"]

## get velocity at base of mixed layer
w_n3 = src.utils.reconstruct_wrapper(
    total[["w", "w_comp"]],
    lambda x: get_w_sub(sel_n3(x)),
)["w"]

## merge results
total_n3 = xr.merge([dTdz_n3.rename("dTdz_n3"), w_n3.rename("w_n3")])

#### $\partial T / \partial x$

In [ ]:
## compute diff. def of Tw
sel_lat = lambda x: x.sel(latitude=slice(-5, 5)).mean(["latitude", "longitude"])
get_Tw = lambda x: sel_lat(x.sel(longitude=slice(140, 190)))
get_Te = lambda x: sel_lat(x.sel(longitude=slice(240, 280)))
get_dTdx = lambda x: get_Tw(x) - get_Te(x)

## compute dTdx
dTdx = src.utils.reconstruct_wrapper(total[["sst", "sst_comp"]], get_dTdx)
dTdx = dTdx.rename({"sst": "dTdx"})

#### $\partial T / \partial y$

In [ ]:
## compute diff. def of Tw
sel_lon = lambda x: x.sel(longitude=slice(170, 250)).mean(["latitude", "longitude"])
get_T = lambda x: sel_lon(x.sel(latitude=slice(-5, 5)))
get_Ts = lambda x: sel_lon(x.sel(latitude=slice(-15, -5)))
get_dTdy = lambda x: get_Ts(x) - get_T(x)

## compute dTdx
dTdy = src.utils.reconstruct_wrapper(total[["sst", "sst_comp"]], get_dTdy)
dTdy = dTdy.rename({"sst": "dTdy"})

#### $\tau_x$

In [ ]:
def get_taux(x):
    """function to compute area-averaged taux"""

    ## specify lon/lat range for averaging
    lon_range = slice(170, 250)
    lat_range = slice(-5, 5)
    idx = dict(longitude=lon_range, latitude=lat_range)

    ## compute
    return x.sel(idx).mean(["longitude", "latitude"])


## compute
taux = src.utils.reconstruct_wrapper(total[["taux", "taux_comp"]], fn=get_taux)

#### $h$ max-gradient stuff

In [ ]:
# ## load tropical SST avg
trop_sst = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/trop_sst.nc"))

## Load T,h (total)
Th_total = xr.open_dataset(DATA_FP / "cesm" / "Th.nc")
Th_total = xr.merge([Th_total, trop_sst])
Th_anom = Th_total - Th_total.mean("member")

## custom h data
h_mg_forced, h_mg_anom = src.utils.load_h_data(max_grad=True, use_wide=USE_WIDE)
h_total = h_mg_forced + h_mg_anom

## compute h stats
h = h_total.sel(longitude=slice(155, 270)).mean("longitude")
h_e = h_total.sel(longitude=slice(240, 270)).mean("longitude")
h_w = h_total.sel(longitude=slice(155, 210)).mean("longitude")
h_idxs = xr.merge([h.rename("h_mg"), (h_w - h_e).rename("dhdx_mg")])

Plot sample

In [ ]:
h_early = h_total.sel(time=slice("1850-01", "1899-12")).mean(["time", "member"])
h_middle = h_total.sel(time=slice("1990-01", "2029-12")).mean(["time", "member"])
h_late = h_total.sel(time=slice("2060-01", "2099-12")).mean(["time", "member"])

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(h_early.longitude, h_early)
ax.plot(h_early.longitude, h_middle)
ax.plot(h_early.longitude, h_late)

## plot lines
ax_kwargs = dict(ls="--", lw=0.8, c="gray")
for t in [155, 210, 240, 270]:
    ax.axvline(t, **ax_kwargs)

## set ylimit
ax.set_ylim(ax.get_ylim()[::-1])

plt.show()

### Combine data

In [ ]:
## specify time index for plotting
## start index: use 3 for SON, 4 for DJF, etc.
time_idx = slice(0, None, None)
# time_idx = slice(0,None,4) ## for JFM

## combine data
data = xr.merge(
    [Th_total, total_n3, h_idxs, taux, dTdx, dTdy, Th_anom["T_34"].rename("T_34a")]
)

## get windowed version
data = src.utils.get_windowed(
    data.isel(time=slice(None, -1)), window_size=480, stride=60
)

## resample to DJF
data = data.resample({"time": "QS-JAN"}).mean().isel(time=time_idx)

## compute ensemble mean and sigma
data_mean = data.mean("time")
data_sigma = data.std("time")

#### Look at subset

In [ ]:
## specify year index
yi = 7

fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")
for ax, v in zip(axs, ["w_n3", "dTdx", "dTdz_n3", "dTdy"]):
    ax.set_title(v)
    ax.scatter(
        data_sigma["T_34a"].isel(year=yi),
        data_mean[v].isel(year=yi),
    )

### Remove nonlinear rect. effect

In [ ]:
## Linear regression object
LR = sklearn.linear_model.LinearRegression

## empty array to hold results
coefs = []

## loop thru years
for year in data.year:

    ## get data
    X = data_sigma["T_34a"].sel(year=year)
    # Y = data_mean["dTdx"].sel(year=year)
    Y = data_mean.sel(year=year).to_dataarray().transpose("member", ...)

    ## instantiate model
    mod = LR(fit_intercept=True)
    mod.fit(X=X.values[:, None], y=Y.values)

    ## get coefficients
    coefs_y = np.concatenate([mod.coef_, mod.intercept_[:, None]], axis=1)

    ## put in xr
    coefs_y = xr.DataArray(
        coefs_y,
        coords=dict(variable=Y["variable"], degree=[1, 0]),
        dims=["variable", "degree"],
    )

    coefs.append(coefs_y.to_dataset("variable"))

## put coefs in array
coefs = xr.concat(coefs, dim=data.year)

## Plot

In [ ]:
## func to get fractional changes
def sel(x, x0=None):
    """get fractional change"""

    ## get baseline if not specified
    if x0 is None:
        x0 = x.isel(year=0)

    ## compute
    return (x - x0) / x0


labels = {
    "taux": r"$\tau_x$",
    "w_n3": r"$w$",
    "dTdx": r"$\partial T/\partial x$",
    "dTdz_n3": r"$\partial T / \partial z$",
    "h_mg": r"$h$",
    "dhdx_mg": r"$\partial h/\partial x$",
    "dTdy": r"$\partial T\partial y$",
}

In [ ]:
## specify variables to plot
plot_vars = ["taux", "w_n3", "dTdx", "dTdz_n3", "h_mg", "dhdx_mg", "dTdy"]

fig, axs = plt.subplots(
    # 1, len(plot_vars), figsize=(2.5 * len(plot_vars), 2.5), layout="constrained"
    2,
    4,
    figsize=(9, 4.5),
    layout="constrained",
)

for ax, v in zip(axs.flatten(), plot_vars):

    ## compute
    data_v_clim = data[v].mean("time")
    median = data_v_clim.quantile(q=0.5, dim="member")
    baseline = median.isel(year=0)

    ## plot each quantile
    for q, lw in zip([0.5, 0.05, 0.95], [2, 1, 1]):

        ## get data for quantile
        data_q = sel(data_v_clim.quantile(q=q, dim="member"), x0=baseline)

        ## plot
        ax.plot(data.year, data_q, c="k", lw=lw)

    ## plot diagnosed
    ax.plot(data.year, sel(coefs.sel(degree=0)[v]))

    ## plot reference line for tau_x
    data_taux_clim = data["taux"].mean("time")
    median_taux = data_taux_clim.quantile(q=0.5, dim="member")
    plot_taux = sel(median_taux)
    if "dz" in v:
        plot_taux *= -1
    ax.plot(data.year, plot_taux, c="magenta", ls="--", lw=1)

    ax_kwargs = dict(c="gray", lw=0.5, ls="--")
    ax.axvline(2010, **ax_kwargs)
    ax.axvline(2085, **dict(ax_kwargs, ls="-"))
    ax.set_title(labels[v])
    ax.axhline(0, **dict(ax_kwargs, ls="-"))

    if "dz" in v:
        ax.set_ylim([-0.1, 0.4])
        ax.axhline(0.1, **ax_kwargs)
    else:
        ax.set_ylim([0.1, -0.4])
        ax.axhline(-0.1, **ax_kwargs)

for ax in axs[:, 1:].flatten():
    ax.set_yticks([])
for ax in axs[0]:
    ax.set_xticks([])
for ax in axs[-1]:
    ax.set_xticks([1870, 2010, 2080])

plt.show()